# FashionMNIST robustness — Lumiere Research

**Status: historical research code; public notebook cleanup.**

This notebook experiments with a fully connected classifier, learning rate, hidden-layer depth, and additive noise. It retains the implemented experiment rather than claiming to reproduce every experiment in the accompanying research work.

Default: four 512-unit hidden layers, SGD with learning rate 0.001, 50 epochs, batch size 64, CPU, clean training and clean testing. Change the two flags below to select noisy datasets. The noise is additive positive uniform noise with strength 0.5, redrawn on each sample access and not clipped to [0,1]. It is not Gaussian noise.

The model/data/training scaffolding is adapted from the [PyTorch Learn the Basics tutorials](https://docs.pytorch.org/tutorials/beginner/basics/intro.html). See the included tutorial license and attribution.


In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt


training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)


## Inspect clean examples


In [ ]:
labels_map = {
    0: "T-Shirt",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle Boot",
}
figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(training_data), size=(1,)).item()
    img, label = training_data[sample_idx] #get item at sampleindex 
    figure.add_subplot(rows, cols, i)
    plt.title(labels_map[label])
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()


## Clean data loaders


In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(training_data, batch_size=64, shuffle=False) # original clean loader uses a fixed training order 
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)

train_features, train_labels = next(iter(train_dataloader)) #chooses one batch
print(f"Feature batch shape: {train_features.size()}") #ex output ([64, 1, 28, 28]), 64 is the batch size, 1 represents black and white image
print(f"Labels batch shape: {train_labels.size()}")
img = train_features[0].squeeze()
label = train_labels[0]
plt.imshow(img, cmap="gray")
plt.show()
print(f"Label: {label}")
print(train_features[0]) #outputs 28 rows/columns with range of values from 0-1, 0 being black and 1 being white


In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import Compose,ToTensor


## Additive uniform-noise datasets


In [ ]:
class AddNoise(object):

    def __init__(self, strength=1):
      self.strength = strength

    def __call__(self, img):
        noise = torch.rand(img.shape)
        return img + self.strength * noise


training_noisedata = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=Compose([ToTensor(), AddNoise(0.5)]) #replace with RandomHorizontalFlip()
)


test_noisedata = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=Compose([ToTensor(), AddNoise(0.5)])
)

train_noisedataloader = DataLoader(training_noisedata, batch_size=64, shuffle=True) #choose 64 randomly in the data set 
test_noisedataloader = DataLoader(test_noisedata, batch_size=64, shuffle=True)


In [ ]:
labels_map = {
    0: "T-Shirt",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle Boot",
}
figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(training_noisedata), size=(1,)).item()
    img, label = training_noisedata[sample_idx] #get item at sampleindex 
    figure.add_subplot(rows, cols, i)
    plt.title(labels_map[label])
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()


## Model and optimization

The hidden-layer count is changed in the architecture below; learning rate is configured in the final training cell.


In [ ]:
# device = (
#     "cuda"
#     if torch.cuda.is_available()
#     else "mps"
#     if torch.backends.mps.is_available()
#     else "cpu"
# )
device="cpu"
print(f"Using {device} device")


In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 512), #can change the number of layers/size of layers and experiment with that
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    correct = 0
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)
        correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        # zero.grad(5,4)

        # if batch % 100 == 0:
        #     loss, current = loss.item(), batch * batch_size + len(X)
        #     print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    correct /= size
    print(f"Train accuracy: {correct*100}")
    return 100*correct



def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return 100*correct


## Select the experiment and train

Noise is sampled anew on each visit. There is no fixed corruption benchmark or automated learning-rate sweep here. The original notebook did not pin seeds, record its exact environment, or separate a validation set for tuning. Avoid using the test set to choose the best hyperparameters when extending this experiment.


In [ ]:
TRAIN_WITH_NOISE = False
TEST_WITH_NOISE = False
selected_train_loader = train_noisedataloader if TRAIN_WITH_NOISE else train_dataloader
selected_test_loader = test_noisedataloader if TEST_WITH_NOISE else test_dataloader
from torch import nn
import torch
learning_rate = 1e-3 #how fast to update paramters in model #can experiement with the learning rate
batch_size = 64 #how many images the machine looks at once
model = NeuralNetwork().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)


epochs = 50 #number of times to pass data through machine
train_accuracies=[]
test_accuracies=[]
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_accuracy=train_loop(selected_train_loader, model, loss_fn, optimizer) #replace train_dataloader with train_noisedataloader, note down the accuracy at the last epoch (50)
    test_accuracy=test_loop(selected_test_loader, model, loss_fn)
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)
print("Done!")


In [ ]:
import matplotlib.pyplot as plt
plt.plot(train_accuracies, label='train accuracy')
plt.plot(test_accuracies, label='test accuracy')
plt.title('Training and Test Accuracy Trajectories')
plt.xlabel('epochs')
plt.ylabel('accuracy')
plt.legend()
plt.show()
